# Tarea 2 · Parte 1 — Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿Pueden convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| Dato | Valor |
|---|---|
| Estudiante | Sandy Patricia Martinez Jara (@sandypmj) |
| Fecha de extracción | 2026-10-24 |
| Versión de PaddleOCR | 3.7.0 |
| Modelo ASR | faster-whisper `small` (int8, CPU, VAD activado) · versión 1.2.1 |
| Diarización | WeSpeaker ResNet34 (pyannote) + agrupamiento jerárquico, vía `voz.diarizar` |

**Fuentes y licencias** (detalle archivo por archivo en `datos/fuentes.csv`):

| Fuente | Modalidad | Licencia |
|---|---|---|
| Wikimedia Commons (documentos y billetes peruanos) | documentos | según cada archivo, leída de la API de Commons |
| Capturas propias con celular, sin datos personales | documentos | CC BY 4.0, autora: la estudiante |
| OpenSLR 73 — Crowdsourced Peruvian Spanish speech (Google) | audio | CC BY-SA 4.0 |
| Entrevistas de Wikimedia Commons (El Agustino, Lima; Fenmucarinap) | audio | CC BY-SA 4.0 |

In [1]:
import json
import time
import warnings
import zipfile
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import soundfile as sf

import datos as datos_curso   # datos.py de la Sesión 13 (descarga por partes de OpenSLR 73)
import utiles                  # nuestras funciones de apoyo
import voz                     # voz.py de la Sesión 13 (VAD, transcripción, diarización)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Carpetas. Todo lo pesado va a datos/raw/ (está en .gitignore: NO se sube a GitHub)
RAW = Path("datos/raw")
RAW_DOC, RAW_AUD = RAW / "documentos", RAW / "audio"
EJEMPLOS = Path("datos/ejemplos")      # muestra ligera y publicable para el dashboard
for c in [RAW_DOC / "commons", RAW_DOC / "ocr_crudo", RAW_AUD / "commons", RAW_AUD / "salidas", EJEMPLOS,
          RAW_DOC / "propios" / "impreso", RAW_DOC / "propios" / "foto", RAW_DOC / "propios" / "manuscrito",
          RAW_AUD / "propios"]:
    c.mkdir(parents=True, exist_ok=True)

FECHA = date.today().isoformat()
SEMILLA = 2026
print("✅ Librerías cargadas y carpetas creadas")

✅ Librerías cargadas y carpetas creadas


In [2]:
import faster_whisper
import paddleocr

print("Fecha de extracción:", FECHA)
print("PaddleOCR:", paddleocr.__version__)
print("faster-whisper:", faster_whisper.__version__)
print("pandas:", pd.__version__)

Fecha de extracción: 2026-10-24
PaddleOCR: 3.7.0
faster-whisper: 1.2.1
pandas: 3.0.6


---
# Parte A · Documentos (25: 8 impresos, 8 fotos, 9 manuscritos)

Dos orígenes:
1. **Wikimedia Commons** (14 archivos): solo escribimos el nombre exacto del archivo; la licencia, el autor y la URL se piden a la API de Commons automáticamente.
2. **Capturas propias** (11 archivos) en `datos/raw/documentos/propios/<tipo>/`: 5 fotos de impresos peruanos con sombra o inclinación y 6 manuscritos, todos sin datos personales. El preprocesamiento aplicado (rotación de las hojas del cuaderno, rectángulos negros sobre el número de tarjeta de una boleta y sobre un código QR) está documentado en `datos/preprocesamiento_propios.csv`.

In [3]:
from PIL import Image

# (nombre exacto en Commons, tipo, descripción, recorte opcional (izq, arriba, der, abajo) en fracciones)
DOCS_COMMONS = [
    ("Acta de Creación de los Símbolos para el Distrito de Mache.jpg", "impreso", "Resolución de alcaldía de Mache (2016)", None),
    ("Pisco-punch-mercurio-peruano-1791.jpg", "impreso", "Mercurio Peruano (1791)", None),
    ("10 Intis 1987(front).jpg", "impreso", "Billete de 10 intis (1987)", None),
    ("50 Intis 1987(Front).jpg", "impreso", "Billete de 50 intis (1987)", None),
    ("100 Intis 1987(Front).jpg", "impreso", "Billete de 100 intis (1987)", None),
    ("1000 Intis 1988(Front).jpg", "impreso", "Billete de 1000 intis (1988)", None),
    ("5000 Intis 1988(Front).jpg", "impreso", "Billete de 5000 intis (1988)", None),
    ("10000 Intis 1988(Front).jpg", "impreso", "Billete de 10 000 intis (1988)", None),
    ("Placa Calle Capon.jpg", "foto", "Placa de la Calle Capón, foto en la calle", None),
    ("Menu Levaggi 09052010.jpg", "foto", "Pizarra de menú con tiza", None),
    ("Cartel Picante de cuy.JPG", "foto", "Cartel «Picante de cuy» pintado a mano", None),
    ("Tarjeta de Ricardo Palma a Luis Antonio Eguiguren Escudero 24 octubre 1912.png", "manuscrito",
     "Tarjeta manuscrita de Ricardo Palma (1912), solo la parte escrita a mano", (0, 0.03, 1, 0.62)),
    ("Acta de Independencia Peru.jpg", "manuscrito", "Acta de la Independencia (1821)", None),
    ("Peruvian Telephone Company receipt Lima 1895.JPG", "manuscrito", "Recibo de teléfono llenado a mano (1895)", None),
]

filas = []
for titulo, tipo, desc, recorte in DOCS_COMMONS:
    info = utiles.info_commons(titulo, ancho=1600)
    sufijo = Path(info["url_descarga"].split("?")[0]).suffix.lower() or ".jpg"
    ruta = utiles.descargar(info["url_descarga"], RAW_DOC / "commons" / (utiles.nombre_seguro(titulo) + sufijo))
    if recorte:  # guardamos el recorte como archivo aparte (respaldo crudo + recorte)
        img = Image.open(ruta).convert("RGB")
        w, h = img.size
        ruta_r = ruta.with_name(ruta.stem + "_recorte.png")
        img.crop((int(recorte[0] * w), int(recorte[1] * h), int(recorte[2] * w), int(recorte[3] * h))).save(ruta_r)
        ruta = ruta_r
    filas.append({"ruta_original": ruta.as_posix(), "tipo": tipo, "descripcion": desc, "origen": "Wikimedia Commons",
                  "fuente": f"Wikimedia Commons ({info['licencia']})", "fuente_url": info["fuente_url"],
                  "licencia": info["licencia"], "autor": info["autor"], "publicable": True})
    print(f"✓ {tipo:10s} {info['licencia']:22s} {titulo}")

for tipo in ["impreso", "foto", "manuscrito"]:
    for ruta in sorted((RAW_DOC / "propios" / tipo).glob("*")):
        if ruta.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            filas.append({"ruta_original": ruta.as_posix(), "tipo": tipo, "descripcion": f"Captura propia: {ruta.stem}",
                          "origen": "Captura propia", "fuente": "Captura propia (CC BY 4.0)",
                          "fuente_url": "captura propia (archivo local, no publicado)", "licencia": "CC BY 4.0",
                          "autor": "estudiante", "publicable": False})

docs = pd.DataFrame(filas)
print("\nDocumentos reunidos por tipo:")
docs["tipo"].value_counts()

✓ impreso    CC BY 4.0              Acta de Creación de los Símbolos para el Distrito de Mache.jpg
✓ impreso    Public domain          Pisco-punch-mercurio-peruano-1791.jpg
✓ impreso    CC0                    10 Intis 1987(front).jpg
✓ impreso    CC0                    50 Intis 1987(Front).jpg
✓ impreso    CC0                    100 Intis 1987(Front).jpg
✓ impreso    CC0                    1000 Intis 1988(Front).jpg
✓ impreso    CC0                    5000 Intis 1988(Front).jpg
✓ impreso    CC0                    10000 Intis 1988(Front).jpg
✓ foto       CC BY 3.0              Placa Calle Capon.jpg
✓ foto       CC BY-SA 3.0           Menu Levaggi 09052010.jpg
✓ foto       Public domain          Cartel Picante de cuy.JPG
✓ manuscrito Public domain          Tarjeta de Ricardo Palma a Luis Antonio Eguiguren Escudero 24 octubre 1912.png
✓ manuscrito Public domain          Acta de Independencia Peru.jpg
✓ manuscrito Public domain          Peruvian Telephone Company receipt Lima 1895.JPG

Doc

tipo
manuscrito    9
impreso       8
foto          8
Name: count, dtype: int64

In [4]:
docs["hash"] = docs["ruta_original"].map(lambda r: utiles.huella(Path(r)))
duplicados = docs[docs.duplicated("hash")]
descartes = [{"archivo": r.ruta_original, "modalidad": "documento", "motivo": "duplicado (mismo hash)"}
             for r in duplicados.itertuples()]
docs = docs.drop_duplicates("hash").reset_index(drop=True)
docs.insert(0, "id", [f"doc_{i:03d}" for i in range(1, len(docs) + 1)])
print(f"Duplicados eliminados: {len(duplicados)} · Documentos únicos: {len(docs)}")

Duplicados eliminados: 0 · Documentos únicos: 25


## OCR con PaddleOCR

**Configuración (definida antes de ver resultados y no se cambia después):**
- `lang="es"`: modelo de reconocimiento para español (tildes y ñ).
- Igual que en la Sesión 10, se desactivan la clasificación de orientación de página, el *unwarping* y la orientación por línea (son lentos en CPU). Por eso las hojas del cuaderno se rotaron antes (ver `preprocesamiento_propios.csv`).
- Imágenes de Commons pedidas a 1600 px de ancho; recorte de la parte manuscrita de la tarjeta de Palma.

Por cada documento se guarda el **respaldo crudo** (JSON con cada línea, su caja y su confianza) y la imagen con las cajas detectadas en `datos/raw/documentos/ocr_crudo/`.

In [8]:
from paddleocr import PaddleOCR

OPCIONES_OCR = dict(lang="es", use_doc_orientation_classify=False, use_doc_unwarping=False,
                    use_textline_orientation=False,
                    enable_mkldnn=False)   # evita un error conocido de Paddle 3.x en Windows/CPU
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (lang=es, sin orientación ni unwarping, mkldnn off)"
ocr = PaddleOCR(**OPCIONES_OCR)
print("✅ PaddleOCR listo")

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\CLIENTE\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\CLIENTE\.paddlex\official_models\PP-OCRv6_medium_rec`.


✅ PaddleOCR listo


In [9]:
textos, confianzas, lineas, segundos = [], [], [], []
for r in docs.itertuples():
    t0 = time.perf_counter()
    res = ocr.predict(r.ruta_original)[0]
    segundos.append(round(time.perf_counter() - t0, 2))
    carpeta = RAW_DOC / "ocr_crudo" / r.id          # respaldo crudo: JSON + imagen con las cajas
    carpeta.mkdir(parents=True, exist_ok=True)
    res.save_to_json(str(carpeta))
    res.save_to_img(str(carpeta))
    textos.append(" ".join(res["rec_texts"]).strip())
    confianzas.append(round(float(np.mean(res["rec_scores"])), 3) if len(res["rec_scores"]) else 0.0)
    lineas.append(len(res["rec_texts"]))
    print(f"{r.id} [{r.tipo:10s}] {lineas[-1]:3d} líneas · confianza {confianzas[-1]:.2f} · {segundos[-1]:5.1f} s")

docs["texto_ocr"], docs["confianza_media"], docs["n_lineas"], docs["segundos_proceso"] = textos, confianzas, lineas, segundos
docs["modelo_ocr"] = MODELO_OCR
print("\n✅ OCR terminado")

Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/fonts/simfang.ttf ...
[==================================================] 100.00%


doc_001 [impreso   ]  28 líneas · confianza 0.99 ·  91.1 s
doc_002 [impreso   ]  44 líneas · confianza 0.94 ·  83.5 s
doc_003 [impreso   ]  12 líneas · confianza 0.93 ·  36.8 s
doc_004 [impreso   ]  14 líneas · confianza 0.89 ·  35.0 s
doc_005 [impreso   ]  15 líneas · confianza 0.91 ·  34.3 s
doc_006 [impreso   ]  10 líneas · confianza 0.92 ·  36.0 s
doc_007 [impreso   ]  12 líneas · confianza 0.92 ·  41.3 s
doc_008 [impreso   ]  11 líneas · confianza 0.92 ·  40.2 s
doc_009 [foto      ]   9 líneas · confianza 0.92 ·  90.0 s
doc_010 [foto      ]  22 líneas · confianza 0.93 ·  62.1 s
doc_011 [foto      ]   0 líneas · confianza 0.00 ·  48.4 s
doc_012 [manuscrito]   7 líneas · confianza 0.73 ·  43.4 s
doc_013 [manuscrito]  46 líneas · confianza 0.60 ·  32.6 s
doc_014 [manuscrito]  17 líneas · confianza 0.85 ·  21.6 s
doc_015 [foto      ]  87 líneas · confianza 1.00 ·  62.9 s
doc_016 [foto      ]  97 líneas · confianza 0.99 ·  97.6 s
doc_017 [foto      ]  29 líneas · confianza 0.97 ·  49.9

Resized image size (3120x4160) exceeds max_side_limit of 4000. Resizing to fit within limit.


doc_021 [manuscrito]  27 líneas · confianza 0.91 · 508.9 s


Resized image size (3120x4160) exceeds max_side_limit of 4000. Resizing to fit within limit.


...

doc_023 [manuscrito]  27 líneas · confianza 0.91 · 357.4 s


Resized image size (3120x4160) exceeds max_side_limit of 4000. Resizing to fit within limit.


doc_024 [manuscrito]  32 líneas · confianza 0.90 · 544.0 s


Resized image size (3120x4160) exceeds max_side_limit of 4000. Resizing to fit within limit.


doc_025 [manuscrito]  31 líneas · confianza 0.88 · 279.7 s

✅ OCR terminado


In [10]:
# Respaldo del OCR (tardó mucho): si se reinicia el kernel, se recupera con pd.read_pickle
docs.to_pickle(RAW / "docs_ocr_respaldo.pkl")
print("💾 Respaldo guardado:", len(docs), "documentos")

💾 Respaldo guardado: 25 documentos


### Documentos ilegibles

**Criterio (fijado antes de mirar resultados):** un documento se considera ilegible si el OCR devuelve menos de 15 caracteres **o** la confianza media es menor a 0.50. Los ilegibles se descartan, se reportan y deben reemplazarse por otro del mismo tipo.

In [11]:
ilegible = (docs["texto_ocr"].str.len() < 15) | (docs["confianza_media"] < 0.50)
for r in docs[ilegible].itertuples():
    descartes.append({"archivo": r.ruta_original, "modalidad": "documento",
                      "motivo": f"ilegible ({len(r.texto_ocr)} caracteres, confianza {r.confianza_media})"})
docs = docs[~ilegible].reset_index(drop=True)

conteo = docs["tipo"].value_counts()
print(conteo.to_string())
faltan = {t: m - conteo.get(t, 0) for t, m in {"impreso": 8, "foto": 8, "manuscrito": 9}.items() if conteo.get(t, 0) < m}
print("✅ Mínimos cumplidos" if not faltan else f"⚠️ FALTAN documentos: {faltan}")

tipo
manuscrito    9
impreso       8
foto          7
⚠️ FALTAN documentos: {'foto': np.int64(1)}


In [12]:
pd.DataFrame(descartes)

,archivo,modalidad,motivo
0,datos/raw/documentos/commons/Cartel_Picante_de_cuy_ac7b64.jpg,documento,"ilegible (0 caracteres, confianza 0.0)"


In [13]:
# Reemplazo del documento ilegible: OCR solo de la foto nueva (no repetimos los 25)
nueva = RAW_DOC / "propios" / "foto" / "foto_06.jpg"
t0 = time.perf_counter()
res = ocr.predict(str(nueva))[0]
seg = round(time.perf_counter() - t0, 2)
carpeta = RAW_DOC / "ocr_crudo" / "doc_026"
carpeta.mkdir(parents=True, exist_ok=True)
res.save_to_json(str(carpeta))
res.save_to_img(str(carpeta))

fila = {"id": "doc_026", "ruta_original": nueva.as_posix(), "tipo": "foto",
        "descripcion": "Captura propia: foto_06 (reemplazo del cartel ilegible)",
        "origen": "Captura propia", "fuente": "Captura propia (CC BY 4.0)",
        "fuente_url": "captura propia (archivo local, no publicado)", "licencia": "CC BY 4.0",
        "autor": "estudiante", "publicable": False, "hash": utiles.huella(nueva),
        "texto_ocr": " ".join(res["rec_texts"]).strip(),
        "confianza_media": round(float(np.mean(res["rec_scores"])), 3) if len(res["rec_scores"]) else 0.0,
        "n_lineas": len(res["rec_texts"]), "segundos_proceso": seg, "modelo_ocr": MODELO_OCR}
print(f"doc_026: {fila['n_lineas']} líneas · confianza {fila['confianza_media']} · {len(fila['texto_ocr'])} caracteres")

if len(fila["texto_ocr"]) >= 15 and fila["confianza_media"] >= 0.50 and fila["hash"] not in set(docs["hash"]):
    docs = pd.concat([docs, pd.DataFrame([fila])], ignore_index=True)
    docs.to_pickle(RAW / "docs_ocr_respaldo.pkl")      # actualizamos el respaldo
    print("✅ Reemplazo aceptado")
else:
    print("⚠️ También ilegible o duplicada: prueba con otra foto")
print(docs["tipo"].value_counts().to_string())

doc_026: 63 líneas · confianza 0.963 · 1125 caracteres
✅ Reemplazo aceptado
tipo
manuscrito    9
impreso       8
foto          8


---
# Parte B · Audio (≥ 30 minutos, ≥ 20 clips, ≥ 1 con 2+ hablantes)

Orígenes, todos con licencia libre:
1. **Entrevistas de Wikimedia Commons** en español peruano (las de la Sesión 13). La *Entrevista 2* de El Agustino tiene **varias personas hablando**: sirve para la diarización.
2. **OpenSLR 73** (español peruano leído, CC BY-SA 4.0): completa los minutos. Cada clip junta frases **de un mismo hablante** (≈ 60 s) separadas por medio segundo de silencio.

Las grabaciones largas se cortan en **clips de hasta 90 s**, siempre en una pausa detectada por el VAD.

In [14]:
AUDIOS_COMMONS = [
    ("Entrevista 1 en la Av. Rivagüero - El Agustino.ogg", "Entrevista en la calle (El Agustino)"),
    ("Entrevista 2 en la Av. Rivagüero - El Agustino.ogg", "Entrevista en la calle a dos vendedoras (El Agustino)"),
    ("Entrevista-Aurora Coronado-organización de mujeres indígenas-fiestas patrias.ogg", "Conferencia de prensa (Fenmucarinap)"),
]
SR = voz.SR
MAX_CLIP = 90.0


def cortar_en_clips(audio, max_seg=MAX_CLIP):
    """Agrupa tramos de voz seguidos hasta ~max_seg y corta en la pausa: devuelve [(inicio, fin)]."""
    tramos = voz.tramos_de_voz(audio, silencio_min=0.5, max_tramo=25)
    clips_t, ini, fin = [], None, None
    for a, b in tramos:
        if ini is None:
            ini, fin = a, b
        elif b - ini <= max_seg:
            fin = b
        else:
            clips_t.append((ini, fin))
            ini, fin = a, b
    if ini is not None:
        clips_t.append((ini, fin))
    return [(max(0, a - 0.5), min(len(audio) / SR, b + 0.5)) for a, b in clips_t]


clips = []
def guardar_clip(x, nombre, **info):
    ruta = RAW_AUD / f"{nombre}.wav"
    sf.write(ruta, x, SR)
    clips.append({"archivo_audio": ruta.as_posix(), "duracion_seg": round(len(x) / SR, 1), **info})


for titulo, desc in AUDIOS_COMMONS:
    info = utiles.info_commons(titulo, ancho=None)
    ruta = utiles.descargar(info["url_descarga"], RAW_AUD / "commons" / (utiles.nombre_seguro(titulo) + ".ogg"))
    audio = voz.cargar_audio(ruta)
    for k, (a, b) in enumerate(cortar_en_clips(audio), 1):
        guardar_clip(audio[int(a * SR):int(b * SR)], f"{ruta.stem[:25]}_{k:02d}",
                     fuente=f"Wikimedia Commons ({info['licencia']})", fuente_url=info["fuente_url"],
                     licencia=info["licencia"], autor=info["autor"], condicion="entrevista en exteriores / con ruido",
                     descripcion=f"{desc}, tramo {k} ({a:.0f}-{b:.0f} s)", multihablante_posible=True, publicable=True)
    print(f"✓ {titulo}: {len(audio) / SR / 60:.1f} min")

print(f"\nHasta ahora: {len(clips)} clips, {sum(c['duracion_seg'] for c in clips) / 60:.1f} minutos")

✓ Entrevista 1 en la Av. Rivagüero - El Agustino.ogg: 2.3 min
✓ Entrevista 2 en la Av. Rivagüero - El Agustino.ogg: 3.2 min
✓ Entrevista-Aurora Coronado-organización de mujeres indígenas-fiestas patrias.ogg: 4.2 min

Hasta ahora: 8 clips, 9.6 minutos


In [15]:
import io
OBJETIVO_SEG, OBJETIVO_CLIPS, SEG_POR_CLIP = 32 * 60, 22, 60
falta_seg = max(0, OBJETIVO_SEG - sum(c["duracion_seg"] for c in clips))
n_clips_openslr = max(OBJETIVO_CLIPS - len(clips), int(np.ceil(falta_seg / SEG_POR_CLIP)))
print(f"Clips de OpenSLR 73 por crear: {n_clips_openslr}")

silencio = np.zeros(int(0.5 * SR), dtype=np.float32)
zips = {s: zipfile.ZipFile(datos_curso.ZipRemoto(datos_curso.OPENSLR73.format(sexo=s))) for s in ["female", "male"]}
tablas = {}
for s, z in zips.items():
    t = pd.DataFrame([l.split("\t") for l in z.read("line_index.tsv").decode().strip().splitlines()], columns=["id", "texto"])
    t["hablante"] = t["id"].str.rsplit("_", n=1).str[0]
    tablas[s] = t

orden = []
for s, t in tablas.items():
    hs = list(t["hablante"].drop_duplicates().sample(frac=1, random_state=SEMILLA))
    orden.append([(s, h) for h in hs])
orden = [x for par in zip(*orden) for x in par]     # alterna mujer / hombre

for (s, h) in orden[:n_clips_openslr]:
    trozos, total = [], 0.0
    for fid in tablas[s].query("hablante == @h")["id"]:
        x, sr0 = sf.read(io.BytesIO(zips[s].read(f"{fid}.wav")), dtype="float32")
        x = voz.cargar_audio((sr0, x))
        trozos += [x, silencio]
        total += len(x) / SR + 0.5
        if total >= SEG_POR_CLIP:
            break
    guardar_clip(np.concatenate(trozos), f"openslr73_{h}",
                 fuente="OpenSLR 73 (CC BY-SA 4.0)", fuente_url="https://www.openslr.org/73/", licencia="CC BY-SA 4.0",
                 autor="Google (Guevara-Rukoz et al., 2020)", condicion="lectura en estudio (limpio)",
                 descripcion=f"Frases leídas por el hablante {h} ({'mujer' if s == 'female' else 'hombre'})",
                 multihablante_posible=False, publicable=True)
    print(f"✓ openslr73_{h}: {total:.0f} s")
    time.sleep(1)

aud = pd.DataFrame(clips)
print(f"\nTotal: {len(aud)} clips, {aud['duracion_seg'].sum() / 60:.1f} minutos")

Clips de OpenSLR 73 por crear: 23
✓ openslr73_pef_03349: 60 s
✓ openslr73_pem_07508: 61 s
✓ openslr73_pef_03034: 63 s
✓ openslr73_pem_00295: 66 s
✓ openslr73_pef_05223: 63 s
✓ openslr73_pem_02436: 62 s
✓ openslr73_pef_01208: 66 s
✓ openslr73_pem_03349: 61 s
✓ openslr73_pef_06136: 65 s
✓ openslr73_pem_08421: 63 s
✓ openslr73_pef_02436: 64 s
✓ openslr73_pem_06136: 66 s
✓ openslr73_pef_08421: 61 s
✓ openslr73_pem_02484: 64 s
✓ openslr73_pef_02121: 63 s
✓ openslr73_pem_07505: 63 s
✓ openslr73_pef_07508: 66 s
✓ openslr73_pem_01208: 64 s
✓ openslr73_pef_09697: 65 s
✓ openslr73_pem_07049: 65 s
✓ openslr73_pef_03397: 67 s
✓ openslr73_pem_03034: 63 s
✓ openslr73_pef_08784: 65 s

Total: 31 clips, 34.1 minutos


In [16]:
aud["hash"] = aud["archivo_audio"].map(lambda r: utiles.huella(Path(r)))
for r in aud[aud.duplicated("hash")].itertuples():
    descartes.append({"archivo": r.archivo_audio, "modalidad": "audio", "motivo": "duplicado (mismo hash)"})
aud = aud.drop_duplicates("hash").reset_index(drop=True)

aud["segundos_con_voz"] = [round(sum(b - a for a, b in voz.tramos_de_voz(voz.cargar_audio(r))), 1) for r in aud["archivo_audio"]]
for r in aud[aud["segundos_con_voz"] < 3].itertuples():
    descartes.append({"archivo": r.archivo_audio, "modalidad": "audio", "motivo": f"sin voz ({r.segundos_con_voz} s de habla)"})
aud = aud[aud["segundos_con_voz"] >= 3].reset_index(drop=True)
aud.insert(0, "id", [f"clip_{i:03d}" for i in range(1, len(aud) + 1)])
aud.to_pickle(RAW / "aud_respaldo.pkl")

ok = len(aud) >= 20 and aud["duracion_seg"].sum() >= 30 * 60
print(f"{len(aud)} clips · {aud['duracion_seg'].sum() / 60:.1f} min ·", "✅ mínimos cumplidos" if ok else "⚠️ faltan clips o minutos")

31 clips · 34.1 min · ✅ mínimos cumplidos


## Transcripción (ASR) con faster-whisper + VAD, y diarización

- **Modelo:** `faster-whisper small` en CPU (int8), que también es la línea base de la Parte 2.
- **VAD activado** (`vad_filter=True` en `voz.Transcriptor.segmentos`): el modelo solo recibe tramos con voz, lo que evita que Whisper "alucine" en silencios o ruido.
- **Idioma fijado en español** (`idioma="es"`).
- **Diarización** con `voz.diarizar` en los clips de entrevistas; en OpenSLR 73 hay un solo hablante por construcción.
- Por cada clip se exportan **SRT**, **TXT** y **CSV** en `datos/raw/audio/salidas/`.

In [17]:
voz.MODELOS["faster-whisper"] = {**voz.MODELOS["faster-whisper"], "hf": "small", "nombre": "faster-whisper small"}
MODELO_ASR = f"faster-whisper {faster_whisper.__version__} small (int8, CPU, VAD on, beam 5, es)"
asr = voz.Transcriptor("faster-whisper", device="cpu")
print(asr, f"· cargado en {asr.segundos_carga:.0f} s")


def texto_con_hablantes(segmentos):
    """Une los segmentos en turnos: '[HABLANTE_1] ... [HABLANTE_2] ...'."""
    if not segmentos or segmentos[0].hablante is None:
        return " ".join(s.texto for s in segmentos).strip()
    turnos = []
    for s in segmentos:
        etiqueta = f"[HABLANTE_{s.hablante[1:]}]"
        if turnos and turnos[-1][0] == etiqueta:
            turnos[-1][1].append(s.texto)
        else:
            turnos.append((etiqueta, [s.texto]))
    return " ".join(f"{e} {' '.join(t)}" for e, t in turnos)

Transcriptor(faster-whisper small en cpu) · cargado en 2 s


In [18]:
PROGRESO = RAW / "asr_progreso.json"     # guarda cada clip apenas termina
hecho = json.loads(PROGRESO.read_text(encoding="utf-8")) if PROGRESO.exists() else {}

for r in aud.itertuples():
    if r.id in hecho:
        continue
    x = voz.cargar_audio(r.archivo_audio)
    t0 = time.perf_counter()
    segs = asr.segmentos(x, idioma="es")
    seg_proc = round(time.perf_counter() - t0, 1)
    if r.multihablante_posible and len(segs) >= 2:
        segs = voz.diarizar(x, segs)
        n = len({s.hablante for s in segs})
    else:
        n = 1
    base = RAW_AUD / "salidas" / r.id
    Path(f"{base}.srt").write_text(voz.a_srt(segs), encoding="utf-8")
    Path(f"{base}.txt").write_text(voz.a_texto(segs), encoding="utf-8")
    voz.a_tabla(segs).to_csv(f"{base}.csv", index=False)
    hecho[r.id] = {"texto_asr": texto_con_hablantes(segs), "n_hablantes": n, "segundos_proceso": seg_proc}
    PROGRESO.write_text(json.dumps(hecho, ensure_ascii=False), encoding="utf-8")
    print(f"{r.id} {r.duracion_seg:5.0f} s · {n} hablante(s) · {seg_proc:6.1f} s de proceso")

aud["texto_asr"] = aud["id"].map(lambda i: hecho[i]["texto_asr"])
aud["n_hablantes"] = aud["id"].map(lambda i: hecho[i]["n_hablantes"])
aud["segundos_proceso"] = aud["id"].map(lambda i: hecho[i]["segundos_proceso"])
aud["idioma"] = "es (español peruano según la fuente)"
aud["modelo_asr"] = MODELO_ASR
aud.to_pickle(RAW / "aud_respaldo.pkl")
print("\n✅ Transcripción terminada · clips con 2+ hablantes:", (aud["n_hablantes"] >= 2).sum())

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

clip_001    80 s · 3 hablante(s) ·   21.2 s de proceso
clip_002    57 s · 3 hablante(s) ·   15.3 s de proceso
clip_003    90 s · 2 hablante(s) ·   34.9 s de proceso
clip_004    90 s · 3 hablante(s) ·   33.0 s de proceso
clip_005    10 s · 2 hablante(s) ·    3.5 s de proceso
clip_006    90 s · 2 hablante(s) ·   43.2 s de proceso
clip_007    80 s · 2 hablante(s) ·   33.1 s de proceso
clip_008    81 s · 2 hablante(s) ·   30.9 s de proceso
clip_009    60 s · 1 hablante(s) ·   16.4 s de proceso
clip_010    60 s · 1 hablante(s) ·   10.9 s de proceso
clip_011    63 s · 1 hablante(s) ·   16.5 s de proceso
clip_012    66 s · 1 hablante(s) ·   16.0 s de proceso
clip_013    63 s · 1 hablante(s) ·   14.4 s de proceso
clip_014    62 s · 1 hablante(s) ·   14.7 s de proceso
clip_015    66 s · 1 hablante(s) ·   13.0 s de proceso
clip_016    61 s · 1 hablante(s) ·    9.5 s de proceso
clip_017    66 s · 1 hablante(s) ·   13.3 s de proceso
clip_018    63 s · 1 hablante(s) ·   11.7 s de proceso
clip_019  